# Classifying Iris Flowers: A Machine Learning Adventure

Welcome to our machine learning adventure! In this notebook, we'll explore the classic Iris flower dataset and build several models to predict the species of an Iris based on its measurements. This project is a great way to learn how to structure a machine learning workflow, as we'll use separate Python files for data setup, visualization, model training, and evaluation.

## 1. Getting Our Hands Dirty: Data Setup

First things first, let's get our data. The `get_data()` function in `setup.py` will download the Iris dataset, split it into training and testing sets, and save them as CSV files in the `data/` directory. This keeps our project nice and organized.

In [ ]:
import setup
setup.get_data()

## 2. A Picture is Worth a Thousand Flowers: Visualizing the Data

Now that we have our data, let's bring it to life with some visualizations! By plotting the data, we can uncover interesting patterns and relationships between the features. We'll use the functions in `visualization.py` to create some insightful plots.

In [ ]:
import visualization
%matplotlib inline

### Seeing the Whole Picture: The Pair Plot

A pair plot is a great way to see how all the features relate to each other. It shows a grid of scatterplots for every pair of features, giving us a quick overview of the data's structure.

In [ ]:
visualization.plot_pairplot()

### Feature by Feature: Histograms

Histograms allow us to look at the distribution of each feature individually. This helps us understand the range and frequency of values for each measurement.

In [ ]:
visualization.plot_histograms()

### Unpacking the Species: Box Plots

Box plots are perfect for comparing the distributions of features across the different Iris species. This will help us see which features are most useful for telling the species apart.

In [ ]:
visualization.plot_boxplots()

## 3. The Brains of the Operation: Training Our Models

Now for the exciting part! We'll train four different machine learning models to classify the Iris flowers based on our training data. These models, imported from `models.py`, will learn the patterns that distinguish one species from another.

In [ ]:
import models

# Train the models
lr_model = models.train_logistic_regression(random_state=42)
knn_model = models.train_knn(n_neighbors=5)
svm_model = models.train_svm(kernel='linear', C=1.0, probability=True, random_state=42)
dt_model = models.train_decision_tree(max_depth=3, random_state=42)

models_dict = {
    'Logistic Regression': lr_model,
    'KNN': knn_model,
    'SVM': svm_model,
    'Decision Tree': dt_model
}

print("Models trained successfully.")

## 4. The Moment of Truth: Evaluating Our Models

Our models have been trained, but how well did they learn? It's time to test them on the unseen test data. We'll use the functions in `testing.py` to evaluate their performance and see how accurately they can classify the Iris flowers.

In [ ]:
import testing
import pandas as pd

results = {}
iris_species = ['setosa', 'versicolor', 'virginica']

for model_name, model in models_dict.items():
    print(f'--- Evaluating {model_name} ---')
    metrics = testing.evaluate_model(model)
    results[model_name] = {
        'accuracy': metrics['accuracy'],
        'precision': metrics['precision'],
        'recall': metrics['recall'],
        'f1_score': metrics['f1_score']
    }
    print(f"  Accuracy: {metrics['accuracy']:.4f}")
    print(f"  Precision: {metrics['precision']:.4f}")
    print(f"  Recall: {metrics['recall']:.4f}")
    print(f"  F1 Score: {metrics['f1_score']:.4f}")
    testing.plot_confusion_matrix(metrics['confusion_matrix'], iris_species, model_name)
    print('
')

## 5. In-Depth Model Performance Analysis

While the raw metrics provide a quantitative measure of performance, a deeper, more qualitative analysis is essential for a comprehensive understanding. In this section, we'll dissect the results, compare the algorithms, and discuss their broader applicability.

The summary table clearly shows that two models—**K-Nearest Neighbors (KNN)** and **Support Vector Machine (SVM)**—achieved flawless performance on the test set with 100% accuracy. This indicates that for this particular training/test split, the data was perfectly separable by these algorithms.

**Logistic Regression** and the **Decision Tree** were not far behind, achieving an impressive accuracy of approximately 97%. Their minor imperfections highlight the subtle differences in how these algorithms operate and perceive the underlying data structure.

In [ ]:
results_df = pd.DataFrame.from_dict(results, orient='index')
results_df

### Visual Performance Comparison

A bar chart provides an intuitive visual summary of our models' performance metrics, making it easy to compare them at a glance. Since the top-performing models are so close to perfect, we will zoom in on the scores between 0.95 and 1.0 to better appreciate the subtle differences.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set plot style
sns.set(style="whitegrid")

# Create the plot
ax = results_df.plot(kind='bar', figsize=(14, 8), rot=0, colormap='viridis')

# Set titles and labels
ax.set_title('Model Performance Comparison', fontsize=16)
ax.set_ylabel('Score', fontsize=12)
ax.set_xlabel('Models', fontsize=12)
ax.legend(title='Metrics', bbox_to_anchor=(1.05, 1), loc='upper left')
ax.set_ylim(0.95, 1.01) # Zoom in on the top scores for better visibility

# Add labels to the bars
for p in ax.patches:
    ax.annotate(f'{p.get_height():.4f}', (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='center', xytext=(0, 10), textcoords='offset points', fontsize=9)

plt.tight_layout()
plt.show()

## 6. Algorithm Deep Dive: A Comparative Study

Choosing a machine learning model is not just about picking the one with the highest accuracy. It involves a trade-off between performance, interpretability, computational cost, and suitability for the problem at hand. Let's explore why our models performed as they did.

### K-Nearest Neighbors (KNN)
- **Performance:** Perfect accuracy (1.0).
- **How It Works:** KNN is a "lazy learner." It doesn't build a model during training; instead, it memorizes the entire training dataset. To make a prediction, it finds the 'k' closest data points (neighbors) in the feature space and assigns the most common class among them.
- **Analysis:** Its perfect score suggests that the Iris species form distinct, well-separated clusters. When a test data point is introduced, its nearest neighbors are all of the correct species.
- **Strengths:**
  - Simple to understand and implement.
  - Naturally handles multi-class problems and complex decision boundaries.
- **Weaknesses:**
  - Can be computationally expensive during prediction, as it must compute distances to all training points.
  - Performance is highly sensitive to the choice of 'k' and the feature scaling, which we handled properly.

### Support Vector Machine (SVM)
- **Performance:** Perfect accuracy (1.0) with a linear kernel.
- **How It Works:** An SVM aims to find the optimal hyperplane that best separates the classes in the feature space. By maximizing the margin (the distance between the hyperplane and the nearest data points from each class), it creates a robust decision boundary.
- **Analysis:** The success of a linear SVM proves that the Iris dataset is linearly separable. The algorithm was able to draw straight lines (or planes in higher dimensions) that perfectly distinguished the three species.
- **Strengths:**
  - Extremely effective for finding optimal linear boundaries.
  - Memory efficient, as it only relies on the "support vectors" (the points closest to the boundary).
  - Can be extended to non-linear problems using the "kernel trick."
- **Weaknesses:**
  - Can be less interpretable than models like Decision Trees.
  - Training can be slow on very large datasets.

### Logistic Regression
- **Performance:** High accuracy (~0.97).
- **How It Works:** Despite its name, Logistic Regression is a classification algorithm. It models the probability that a given data point belongs to a certain class by fitting a linear equation to the log-odds of the event.
- **Analysis:** Its high performance confirms the largely linear nature of the data. The few misclassifications likely occurred at the boundary between 'versicolor' and 'virginica', where the classes are known to have some overlap.
- **Strengths:**
  - Provides probabilities for predictions, which can be valuable.
  - The model's coefficients offer excellent interpretability, showing how each feature impacts the prediction.
  - Computationally efficient and fast to train.
- **Weaknesses:**
  - Assumes a linear relationship between the features and the outcome, which may not hold for complex problems.

### Decision Tree
- **Performance:** High accuracy (~0.97).
- **How It Works:** A Decision Tree builds a flowchart-like structure of "if-then-else" rules based on the feature values. It recursively splits the data to create the purest possible leaf nodes, which represent the class labels.
- **Analysis:** Like Logistic Regression, its minor errors likely happened where the classes are not perfectly distinct. We limited the tree's depth to 3 to prevent overfitting, which is a crucial step. This simple, interpretable model was still powerful enough to capture the data's structure effectively.
- **Strengths:**
  - Highly interpretable and easy to visualize. The decision-making process is transparent.
  - Requires minimal data preprocessing (e.g., no need for feature scaling).
- **Weaknesses:**
  - Prone to overfitting if not constrained (e.g., by limiting depth).
  - Can be unstable, meaning small changes in the data can lead to a completely different tree.

## 7. Final Verdict and Future Directions

### Which Model is "Best"?

For the Iris dataset, **SVM and KNN are the clear winners in terms of raw predictive accuracy.**

However, the "best" model is context-dependent:
- If **interpretability** is the top priority (e.g., explaining the classification logic to stakeholders), the **Decision Tree** is an excellent choice. Its rules are explicit and easy to understand.
- If we need **probabilistic outputs** and a model that is both interpretable and fast, **Logistic Regression** is a strong contender.
- If **maximum predictive power** is the sole goal for a linearly separable problem like this one, the **linear SVM** is arguably the most robust and theoretically sound choice.

### Future Work and Improvements

This project provides a solid foundation, but there are many ways to expand upon it:
1.  **Hyperparameter Tuning:** We used default or simple parameters (e.g., `k=5`). A more rigorous approach would involve using techniques like **GridSearchCV** or **RandomizedSearchCV** to find the optimal hyperparameters for each model, which could further boost performance.
2.  **Ensemble Methods:** Explore more advanced models like **Random Forests** (an ensemble of Decision Trees) or **Gradient Boosting**, which often yield higher accuracy by combining the predictions of multiple weaker models.
3.  **Cross-Validation:** While we used a simple train-test split, a more robust evaluation technique like **k-fold cross-validation** would provide a more reliable estimate of how the models would perform on unseen data.
4.  **Feature Engineering:** Investigate the creation of new features from the existing ones (e.g., petal area = petal length * petal width) to see if they can improve model performance.

This comprehensive analysis demonstrates a complete machine learning workflow, from data exploration and model training to rigorous evaluation and comparative study.